In [102]:
import pandas as pd

df = pd.read_csv("diabetes_prediction_dataset.csv")

In [103]:
df.head(5)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [104]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  str    
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  str    
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 6.9 MB


In [105]:
# Удаляем дубликаты строк (если есть)
print(f"Rows before deduplication: {len(df)}")
df = df.drop_duplicates().copy()
print(f"Rows after deduplication:  {len(df)}")

# Нормализуем текстовые признаки (убираем лишние пробелы)
df["gender"] = df["gender"].astype(str).str.strip()
df["smoking_history"] = df["smoking_history"].astype(str).str.strip()


Rows before deduplication: 100000
Rows after deduplication:  96146


In [106]:
nan_matrix = df.isnull()
print(nan_matrix.sum())


gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64


In [107]:
for col in df.columns:
    number_unique = df[col].nunique()
    print(f"{col} : {number_unique}")


gender : 3
age : 102
hypertension : 2
heart_disease : 2
smoking_history : 6
bmi : 4247
HbA1c_level : 18
blood_glucose_level : 18
diabetes : 2


In [108]:
# Посмотреть все уникальные значения + частоты (включая NaN)
for col in ["gender", "smoking_history"]:
    print(f"\n=== {col} (value_counts) ===")
    print(df[col].value_counts(dropna=False))


=== gender (value_counts) ===
gender
Female    56161
Male      39967
Other        18
Name: count, dtype: int64

=== smoking_history (value_counts) ===
smoking_history
never          34398
No Info        32887
former          9299
current         9197
not current     6367
ever            3998
Name: count, dtype: int64


In [109]:
# Min / max для числовых столбцов перед обработкой выбросов
num_cols = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]

min_max = df[num_cols].agg(["min", "max"]).T
min_max.columns = ["min", "max"]

print(min_max)

                       min     max
age                   0.08   80.00
bmi                  10.01   95.69
HbA1c_level           3.50    9.00
blood_glucose_level  80.00  300.00


In [110]:
# Биннинг числовых признаков (как дополнительные фичи)
df["Age_Bin"] = pd.qcut(df["age"], q=10, labels=False, duplicates="drop")
df["BMI_Bin"] = pd.qcut(df["bmi"], q=10, labels=False, duplicates="drop")
df["HbA1c_Bin"] = pd.qcut(df["HbA1c_level"], q=6, labels=False, duplicates="drop")
df["Glucose_Bin"] = pd.qcut(df["blood_glucose_level"], q=6, labels=False, duplicates="drop")


In [111]:
# Бинарные признаки переводим в 0/1 для модели
bool_cols = ["hypertension", "heart_disease", "diabetes"]
df[bool_cols] = df[bool_cols].astype(int)


In [112]:
df = pd.get_dummies(
    df,
    columns=["gender", "smoking_history"],
    dtype=int,
    drop_first=True,
)

raw_num_cols = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]
df = df.drop(columns=raw_num_cols)

df.head(5)


,hypertension,heart_disease,diabetes,Age_Bin,BMI_Bin,HbA1c_Bin,Glucose_Bin,gender_Male,gender_Other,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current
0,0,1,0,9,3,5,2,0,0,0,0,0,1,0
1,0,0,0,6,4,5,0,0,0,0,0,0,0,0
2,0,0,0,2,4,2,3,1,0,0,0,0,1,0
3,0,0,0,3,2,1,3,0,0,1,0,0,0,0
4,1,1,0,9,1,1,3,1,0,1,0,0,0,0


In [113]:
df.info()

<class 'pandas.DataFrame'>
Index: 96146 entries, 0 to 99999
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   hypertension                 96146 non-null  int64
 1   heart_disease                96146 non-null  int64
 2   diabetes                     96146 non-null  int64
 3   Age_Bin                      96146 non-null  int64
 4   BMI_Bin                      96146 non-null  int64
 5   HbA1c_Bin                    96146 non-null  int64
 6   Glucose_Bin                  96146 non-null  int64
 7   gender_Male                  96146 non-null  int64
 8   gender_Other                 96146 non-null  int64
 9   smoking_history_current      96146 non-null  int64
 10  smoking_history_ever         96146 non-null  int64
 11  smoking_history_former       96146 non-null  int64
 12  smoking_history_never        96146 non-null  int64
 13  smoking_history_not current  96146 non-null  int64
dtypes: int

In [114]:
for col in df.columns:
    number_unique = df[col].nunique()
    print(f"{col} : {number_unique}")

hypertension : 2
heart_disease : 2
diabetes : 2
Age_Bin : 10
BMI_Bin : 9
HbA1c_Bin : 6
Glucose_Bin : 6
gender_Male : 2
gender_Other : 2
smoking_history_current : 2
smoking_history_ever : 2
smoking_history_former : 2
smoking_history_never : 2
smoking_history_not current : 2


In [115]:
df.to_csv('processed_data.csv', index=False)
